In [1]:
import torch
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"

model_id = "LDKSolutions/KR-patent-deberta-large"

from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("LDKSolutions/KR-patent-deberta-large")
model = AutoModelForMaskedLM.from_pretrained("LDKSolutions/KR-patent-deberta-large")

Loading weights:   0%|          | 0/397 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie deberta.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
DebertaV2ForMaskedLM LOAD REPORT from: LDKSolutions/KR-patent-deberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
)


In [3]:
from datasets import load_dataset

raw_dataset = load_dataset("Dzeniks/wikipedia_keywords", split="train")
split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)
test_valid_split = split_dataset['test'].train_test_split(test_size=0.5, seed=42)

from datasets import DatasetDict

dataset = DatasetDict({
    'train': split_dataset['train'],
    'test': test_valid_split['test'],
    'validation': test_valid_split['train']
})

def tokenize_and_align_labels(examples):
    texts = [str(t) if t is not None else "" for t in examples["keywords"]]
    
    tokenized_inputs = tokenizer(
        texts, 
        truncation=True, 
        padding="max_length", 
        max_length=512
    )
    
    labels = []
    for i, doc_keywords in enumerate(examples["keywords"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        current_keywords = doc_keywords if doc_keywords is not None else []
        current_text = texts[i]
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                is_keyword = any(str(kw) in current_text for kw in current_keywords if kw)
                label_ids.append(1 if is_keyword else 0)
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 데이터셋 필터링 (텍스트가 아예 없는 행 제거)
dataset = dataset.filter(
    lambda x: x['keywords'] is not None and len(str(x['keywords']).strip()) > 0
)

tokenized_ds = dataset.map(
    tokenize_and_align_labels, 
    batched=True,
    remove_columns=dataset['train'].column_names # 원본 텍스트 컬럼 제거 (선택 사항)
)
model = AutoModelForTokenClassification.from_pretrained(model_id, num_labels=2)


Using the latest cached version of the dataset since Dzeniks/wikipedia_keywords couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at /home/lm/.cache/huggingface/datasets/Dzeniks___wikipedia_keywords/default/0.0.0/9e1bc80269bca86277d6ecae90950a6ef8dad23b (last modified on Wed Mar  4 02:31:22 2026).


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2ForTokenClassification LOAD REPORT from: LDKSolutions/KR-patent-deberta-large
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
deberta.embeddings.position_ids            | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. 

In [ ]:
from transformers import TrainingArguments
from transformers import Trainer, DataCollatorForTokenClassification
import os

training_args = TrainingArguments(
    output_dir="./keyword_results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'], 
    data_collator=DataCollatorForTokenClassification(tokenizer),
)

trainer.train()


Step,Training Loss
500,0.033741
1000,0.001400
1500,0.006988
2000,0.000003
2500,0.004997
3000,0.007024
3500,0.001454
4000,0.010566
4500,0.003474
5000,0.000024


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
save_directory = "./my_keyword_model"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"모델이 {save_directory}에 저장되었습니다.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 파일들이 저장된 디렉토리 경로
model_path = "/home/sample/keyword_results/checkpoint-6000/"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("모델 로드 완료")

In [5]:
from transformers import pipeline

model.eval()

nlp = pipeline(
    "token-classification", 
    model=model, 
    tokenizer=tokenizer, 
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

test_text = "Artificial intelligence and machine learning are transforming the world."

results = nlp(test_text)

print(f"입력 문장: {test_text}")
print("-" * 30)
print("추출된 키워드:")
for entity in results:
    if entity['entity_group'] == 'LABEL_1': 
        print(f"- {entity['word']} (신뢰도: {entity['score']:.4f})")

입력 문장: Artificial intelligence and machine learning are transforming the world.
------------------------------
추출된 키워드:
- Artificial intelligence and machine learning are transforming the world. (신뢰도: 1.0000)
